# CCHP Kaggle 入口 `main.ipynb`

用于在 Kaggle 直接运行 `cchp_physical_env` 的训练/评估流程，并支持导入上次导出的 checkpoint 续训。

## Add data 约定
- 代码数据集：包含 `pyproject.toml` + `src/cchp_physical_env/`
- 数据数据集：包含 `data/processed/cchp_main_15min_2024.csv` 与 `data/processed/cchp_main_15min_2025.csv`

按 cell 顺序执行即可。

In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import sys

print('Python:', sys.version)
print('CWD:', Path.cwd())
print('Has /kaggle/input:', Path('/kaggle/input').exists())
print('Has /kaggle/working:', Path('/kaggle/working').exists())
print('CPU count:', os.cpu_count())

for key in ['OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'NUMEXPR_NUM_THREADS']:
    if key in os.environ:
        print(f'{key}:', os.environ.get(key))


## 0) Notebook 参数区（只改这一格）

Notebook 参数优先级最高：后续会生成 runtime config，CLI 全部使用 runtime config（Notebook > base config）。

In [ ]:
from dataclasses import dataclass, field


@dataclass
class RunCfg:
    # Kaggle 数据集目录（你上传的数据集路径）
    kaggle_codeset_dir: str = '/kaggle/input/datasets/tttwwwjjj/cchp-code'
    kaggle_dataset_dir: str = '/kaggle/input/datasets/tttwwwjjj/cchp-data'

    # 上一次导出的输出数据集（用于断点续训）
    # 为空则自动扫描 /kaggle/input
    kaggle_prev_output_dir: str = '/kaggle/input/datasets/tttwwwjjj/cchp-continue'

    # 同步开关
    sync_code_to_cwd: bool = True
    sync_data_to_repo: bool = True

    # 运行配置
    env_config_relpath: str = 'src/cchp_physical_env/config/config.yaml'
    run_root: str = 'runs'
    run_prefix: str = 'kaggle_cchp'

    # 断点续训
    resume_enabled: bool = True
    skip_train_when_resuming: bool = True
    resume_sequence_via_api: bool = True

    # 是否把上次 runs 目录拷贝到当前 runs（通常不需要）
    resume_copy_runs: bool = False

    # 安装最小兜底依赖（避免 Kaggle 环境缺包导致 import 失败）
    install_extra_deps: tuple[str, ...] = (
        'numpy',
        'pandas',
        'pyyaml',
        'pyomo',
        'tespy',
        'CoolProp',
        'transformers',
        'torch',
    )

    # ------------------------------
    # Notebook 只改这一格：覆盖参数
    # 仅通过 overrides 覆盖你要改的键
    # ------------------------------

    # env 覆盖参数（一般不建议改；除非你明确要切换约束模式/后端）
    env_overrides: dict[str, object] = field(default_factory=lambda: {
        # 'constraint_mode': 'physics_in_loop',
        # 'physics_backend': 'tespy',
    })

    # training 覆盖参数（默认与 config.yaml 保持一致：理想初始训练状态）
    # 你最常调的关键参数：policy / sequence_adapter / episode_days / episodes / train_steps / lr / device
    training_overrides: dict[str, object] = field(default_factory=lambda: {
        'seed': 42,
        'policy': 'rule',
        'sequence_adapter': 'rule',
        'history_steps': 16,
        'episode_days': 14,
        'episodes': 800,
        'train_steps': 4096,
        'batch_size': 128,
        'update_epochs': 4,
        'lr': 0.0003,
        'device': 'auto',
        # 序列策略示例（需要时再打开）：
        # 'policy': 'sequence_rule',
        # 'sequence_adapter': 'transformer',
    })

    # eval CLI 覆盖参数
    eval_cli_overrides: dict[str, object] = field(default_factory=lambda: {
        'seed': 42,
    })


rc = RunCfg()
print(rc)


## 1) 同步代码 + 安装

- 扫描 `/kaggle/input` 中包含 `pyproject.toml` + `src/cchp_physical_env` 的目录
- 同步到当前可写目录
- `%pip install -e . --no-deps`
- 注入 `./src` 到 `sys.path`，保证 `import cchp_physical_env` 稳定

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys


def _run_cmd(cmd: list[str]) -> None:
    print('>>', ' '.join(cmd))
    subprocess.run(cmd, check=True)


def _rsync_or_copy(src: Path, dst: Path) -> None:
    src = src.resolve()
    dst = dst.resolve()
    if shutil.which('rsync'):
        _run_cmd(['rsync', '-a', '--info=progress2', f'{src.as_posix()}/', f'{dst.as_posix()}/'])
        return

    print('rsync not found, fallback to shutil.copytree')
    for item in src.iterdir():
        target = dst / item.name
        if item.is_dir():
            if target.exists():
                shutil.rmtree(target)
            shutil.copytree(item, target)
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(item, target)


def _find_codeset_dir(preferred: str) -> Path:
    p = Path(preferred)
    if (p / 'pyproject.toml').exists() and (p / 'src' / 'cchp_physical_env').exists():
        return p

    root = Path('/kaggle/input')
    if not root.exists():
        return p

    hits: list[Path] = []
    for pyproj in root.rglob('pyproject.toml'):
        cand = pyproj.parent
        if (cand / 'src' / 'cchp_physical_env').exists():
            hits.append(cand)

    if len(hits) == 1:
        return hits[0]
    if len(hits) > 1:
        chosen = sorted(hits, key=lambda x: x.as_posix())[0]
        print('Multiple code datasets detected. Using:', chosen)
        return chosen

    return p


code_src = _find_codeset_dir(rc.kaggle_codeset_dir)
assert code_src.exists(), f'missing code dataset: {code_src}'
print('code_src:', code_src)

if rc.sync_code_to_cwd:
    _rsync_or_copy(code_src, Path.cwd())

assert Path('pyproject.toml').exists(), 'missing pyproject.toml after sync'
assert Path('src/cchp_physical_env').exists(), 'missing src/cchp_physical_env after sync'

src_dir = str((Path.cwd() / 'src').resolve())
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
print('sys.executable:', sys.executable)
print('sys.path[0]:', sys.path[0])

ip = get_ipython()
assert ip is not None
ip.run_line_magic('pip', 'install -q -e . --no-deps')
if rc.install_extra_deps:
    ip.run_line_magic('pip', f"install -q {' '.join(rc.install_extra_deps)}")

import cchp_physical_env
print('cchp_physical_env imported from:', cchp_physical_env.__file__)


## 2) 同步数据到 `./data/`

- 扫描 `/kaggle/input` 中包含 `data/processed/*.csv` 的目录
- 同步到 `./data/`
- 强校验 `cchp_main_15min_2024.csv` 与 `cchp_main_15min_2025.csv`

In [ ]:
from pathlib import Path


def _find_data_root(preferred: str) -> Path:
    p = Path(preferred)
    if (p / 'data' / 'processed').exists() and list((p / 'data' / 'processed').glob('*.csv')):
        return p

    root = Path('/kaggle/input')
    if not root.exists():
        return p

    hits: list[Path] = []
    for processed in root.rglob('data/processed'):
        if processed.is_dir() and list(processed.glob('*.csv')):
            hits.append(processed.parent.parent)

    if len(hits) == 1:
        return hits[0]
    if len(hits) > 1:
        chosen = sorted(hits, key=lambda x: x.as_posix())[0]
        print('Multiple data datasets detected. Using:', chosen)
        return chosen

    return p


data_root = _find_data_root(rc.kaggle_dataset_dir)
assert data_root.exists(), f'missing data dataset: {data_root}'
print('data_root:', data_root)

if rc.sync_data_to_repo:
    Path('data').mkdir(parents=True, exist_ok=True)
    _rsync_or_copy(data_root / 'data', Path('data'))

train_csv = Path('data/processed/cchp_main_15min_2024.csv')
eval_csv = Path('data/processed/cchp_main_15min_2025.csv')
assert train_csv.exists(), train_csv
assert eval_csv.exists(), eval_csv
print('train_csv:', train_csv)
print('eval_csv:', eval_csv)


## 3) 生成 runtime config（Notebook 优先级最高）

- 读取 base `config.yaml`
- 仅用 `env_overrides` / `training_overrides` 覆盖你要改的键
- 生成 runtime yaml，后续 CLI 全部使用 runtime yaml

In [ ]:
from pathlib import Path
import copy
import json
import yaml

base_cfg_path = Path(rc.env_config_relpath)
assert base_cfg_path.exists(), base_cfg_path

with base_cfg_path.open('r', encoding='utf-8') as f:
    base_cfg = yaml.safe_load(f)

assert isinstance(base_cfg, dict), 'config.yaml must be a mapping'
assert isinstance(base_cfg.get('env'), dict), 'config.yaml missing env block'
assert isinstance(base_cfg.get('training'), dict), 'config.yaml missing training block'

runtime_cfg = copy.deepcopy(base_cfg)


def _apply_overrides(block_name: str, block: dict, overrides: dict[str, object]) -> None:
    for key, value in overrides.items():
        if key not in block:
            raise KeyError(f'unknown key in notebook overrides: {block_name}.{key}')
        block[key] = value


_apply_overrides('env', runtime_cfg['env'], rc.env_overrides)
_apply_overrides('training', runtime_cfg['training'], rc.training_overrides)

runtime_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
runtime_cfg_path = runtime_dir / f'{rc.run_prefix}_runtime_config.yaml'
runtime_cfg_path.write_text(yaml.safe_dump(runtime_cfg, allow_unicode=True, sort_keys=False), encoding='utf-8')

print('runtime config:', runtime_cfg_path)
print('env_overrides:', json.dumps(rc.env_overrides, ensure_ascii=False, indent=2))
print('training_overrides:', json.dumps(rc.training_overrides, ensure_ascii=False, indent=2))


## 4) 断点续训：自动定位上次 checkpoint

- 扫描 `/kaggle/input`（或你指定的 `kaggle_prev_output_dir`）
- 定位最新的 `baseline_policy.json`
- rule/random：可直接复用 checkpoint，跳过最小 train
- sequence_rule + transformer/mamba：尝试加载上次模型权重 warm-start；失败自动回退 CLI train
- 最终统一走 CLI `eval --checkpoint ...`

In [ ]:
from pathlib import Path
import json
import shutil


def _detect_prev_output_dir(preferred: str) -> Path | None:
    if preferred:
        p = Path(preferred)
        if p.exists():
            return p

    # Kaggle: 用户常用的 continue 数据集位置（存在则优先用）
    continue_dir = Path('/kaggle/input/datasets/tttwwwjjj/cchp-continue')
    if continue_dir.exists():
        return continue_dir

    root = Path('/kaggle/input')
    if not root.exists():
        return None

    candidates: list[Path] = []
    for d in root.rglob('*'):
        if not d.is_dir():
            continue
        if (d / 'kaggle_exports').exists() or (d / 'runs').exists() or (d / 'checkpoints').exists():
            candidates.append(d)

    if not candidates:
        return None

    return sorted(candidates, key=lambda x: x.as_posix())[0]


def _collect_checkpoint_jsons(search_root: Path) -> list[Path]:
    if not search_root.exists():
        return []
    hits = [p for p in search_root.rglob('baseline_policy.json') if p.is_file()]
    return sorted(hits, key=lambda p: p.stat().st_mtime, reverse=True)


resume_checkpoint_json: Path | None = None
prev_output_dir: Path | None = None

if rc.resume_enabled:
    prev_output_dir = _detect_prev_output_dir(rc.kaggle_prev_output_dir)
    print('prev_output_dir:', prev_output_dir)
    if prev_output_dir is not None:
        ckpt_hits = _collect_checkpoint_jsons(prev_output_dir)
        if ckpt_hits:
            resume_checkpoint_json = ckpt_hits[0]
            print('resume checkpoint found:', resume_checkpoint_json)

            if rc.resume_copy_runs:
                run_root = Path(rc.run_root)
                run_root.mkdir(parents=True, exist_ok=True)
                src_runs = prev_output_dir / 'runs'
                if src_runs.exists() and src_runs.is_dir():
                    _rsync_or_copy(src_runs, run_root)
        else:
            print('No baseline_policy.json found under previous output dir.')


## 5) CLI：`summary` / `train` / `eval`

执行顺序：
- `summary`
- 最小 `train`（或 resume 跳过/续训）
- `eval`

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import json


def _run_cli(args: list[str]) -> None:
    env = os.environ.copy()
    py_src = str((Path.cwd() / 'src').resolve())
    env['PYTHONPATH'] = py_src + (os.pathsep + env['PYTHONPATH'] if 'PYTHONPATH' in env else '')
    cmd = [sys.executable, '-m', 'cchp_physical_env', *args]
    print('>>', ' '.join(cmd))
    subprocess.run(cmd, check=True, env=env)


def _latest_checkpoint_from_run_root(run_root: Path) -> Path:
    if not run_root.exists():
        raise FileNotFoundError(f'run_root not found: {run_root}')
    run_dirs = [p for p in run_root.iterdir() if p.is_dir()]
    if not run_dirs:
        raise FileNotFoundError(f'no run dirs under: {run_root}')
    latest = sorted(run_dirs, key=lambda p: p.name)[-1]
    ckpt = latest / 'checkpoints' / 'baseline_policy.json'
    if not ckpt.exists():
        raise FileNotFoundError(f'checkpoint missing: {ckpt}')
    return ckpt


def _resolve_model_checkpoint_from_json(ckpt_json_path: Path, payload: dict) -> Path | None:
    candidates: list[Path] = []
    raw = str(payload.get('model_checkpoint_path', '')).strip()
    if raw:
        candidates.append(Path(raw))
        candidates.append(ckpt_json_path.parent / Path(raw).name)
    candidates.append(ckpt_json_path.parent / 'sequence_policy.pt')
    candidates.append(ckpt_json_path.parent.parent / 'sequence_policy.pt')

    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None


def _try_resume_sequence_training(
    *,
    checkpoint_json_path: Path,
    runtime_config_path: Path,
    run_root: Path,
) -> Path | None:
    try:
        from cchp_physical_env.core.config_loader import (
            build_env_config_from_overrides,
            build_training_options,
            load_env_overrides,
            load_training_overrides,
        )
        from cchp_physical_env.core.data import compute_training_statistics, load_exogenous_data
        from cchp_physical_env.policy.checkpoint import load_policy
        from cchp_physical_env.policy.trainer import (
            SequencePolicyTrainer,
            SequenceTrainerConfig,
        )

        training_options = build_training_options(load_training_overrides(runtime_config_path))
        policy_name = str(training_options.get('policy', 'rule')).strip().lower()
        adapter = str(training_options.get('sequence_adapter', 'rule')).strip().lower()
        if policy_name != 'sequence_rule' or adapter not in {'transformer', 'mamba'}:
            return None

        ckpt_payload = json.loads(checkpoint_json_path.read_text(encoding='utf-8'))
        model_ckpt = _resolve_model_checkpoint_from_json(checkpoint_json_path, ckpt_payload)
        if model_ckpt is None:
            print('resume skipped: sequence model checkpoint not found near', checkpoint_json_path)
            return None

        train_df = load_exogenous_data(Path('data/processed/cchp_main_15min_2024.csv'))
        train_stats = compute_training_statistics(train_df)
        env_config = build_env_config_from_overrides(load_env_overrides(runtime_config_path))

        trainer_cfg = SequenceTrainerConfig(
            policy_backbone=adapter,
            history_steps=int(training_options['history_steps']),
            episode_days=int(training_options['episode_days']),
            train_steps=int(training_options['train_steps']),
            batch_size=int(training_options['batch_size']),
            update_epochs=int(training_options['update_epochs']),
            lr=float(training_options['lr']),
            seed=int(training_options['seed']),
            device=str(training_options['device']),
        )

        trainer = SequencePolicyTrainer(
            train_df=train_df,
            train_statistics=train_stats,
            env_config=env_config,
            config=trainer_cfg,
            run_root=run_root,
        )

        payload = load_policy(checkpoint_path=model_ckpt, map_location=trainer.device)
        trainer.model.load_state_dict(payload['state_dict'], strict=True)
        summary = trainer.train()
        resumed_ckpt_json = Path(summary['checkpoint_json_path'])
        print('resume sequence training done. new checkpoint:', resumed_ckpt_json)
        return resumed_ckpt_json
    except Exception as error:
        print('resume sequence training failed, fallback to CLI train:', repr(error))
        return None


# 1)
_run_cli(['summary', '--env-config', runtime_cfg_path.as_posix()])

# 2)
from cchp_physical_env.core.config_loader import build_training_options, load_training_overrides
training_options = build_training_options(load_training_overrides(runtime_cfg_path))
policy_name = str(training_options.get('policy', 'rule')).strip().lower()
adapter_name = str(training_options.get('sequence_adapter', 'rule')).strip().lower()

active_checkpoint_json: Path | None = None
run_root = Path(rc.run_root)
run_root.mkdir(parents=True, exist_ok=True)

if (
    rc.resume_enabled
    and rc.resume_sequence_via_api
    and resume_checkpoint_json is not None
    and policy_name == 'sequence_rule'
    and adapter_name in {'transformer', 'mamba'}
):
    active_checkpoint_json = _try_resume_sequence_training(
        checkpoint_json_path=resume_checkpoint_json,
        runtime_config_path=runtime_cfg_path,
        run_root=run_root,
    )

if active_checkpoint_json is None:
    if rc.resume_enabled and rc.skip_train_when_resuming and resume_checkpoint_json is not None:
        print('resume checkpoint found, skip fresh train:', resume_checkpoint_json)
        active_checkpoint_json = resume_checkpoint_json
    else:
        _run_cli([
            'train',
            '--env-config', runtime_cfg_path.as_posix(),
            '--run-root', run_root.as_posix(),
        ])
        active_checkpoint_json = _latest_checkpoint_from_run_root(run_root)

assert active_checkpoint_json is not None and active_checkpoint_json.exists(), active_checkpoint_json
print('checkpoint for eval:', active_checkpoint_json)

# 3) 
eval_args = [
    'eval',
    '--env-config', runtime_cfg_path.as_posix(),
    '--checkpoint', active_checkpoint_json.as_posix(),
]
for key, value in rc.eval_cli_overrides.items():
    eval_args.extend([f'--{key.replace("_", "-")}', str(value)])
_run_cli(eval_args)


## 6) 输出检查

- `*_runtime_config.yaml` 是否生成
- `runs/.../checkpoints/baseline_policy.json` 是否存在
- `runs/.../eval/summary.json` 是否存在

In [ ]:
from pathlib import Path

runtime_cfg_candidates = sorted(Path('/kaggle/working').glob('*_runtime_config.yaml')) if Path('/kaggle/working').exists() else []
print('runtime configs:', [p.as_posix() for p in runtime_cfg_candidates])

run_root = Path(rc.run_root)
if run_root.exists():
    run_dirs = sorted([p for p in run_root.iterdir() if p.is_dir()], key=lambda p: p.name)
    print('run dirs:', [p.name for p in run_dirs[-5:]])
    if run_dirs:
        latest = run_dirs[-1]
        print('latest run:', latest)
        print('checkpoint exists:', (latest / 'checkpoints' / 'baseline_policy.json').exists())
        print('eval summary exists:', (latest / 'eval' / 'summary.json').exists())


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import shutil
import time


def _safe_copy(src: Path, dst: Path) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)


export_root = Path('kaggle_exports') / rc.run_prefix
export_root.mkdir(parents=True, exist_ok=True)

export_runtime_cfg = export_root / runtime_cfg_path.name
_safe_copy(runtime_cfg_path, export_runtime_cfg)

export_checkpoint_json = export_root / 'baseline_policy.json'
_safe_copy(active_checkpoint_json, export_checkpoint_json)

run_dir: Path | None = None
try:
    run_dir = active_checkpoint_json.parent.parent
except Exception:
    run_dir = None

export_eval_summary: Path | None = None
if run_dir is not None:
    eval_summary = run_dir / 'eval' / 'summary.json'
    if eval_summary.exists():
        export_eval_summary = export_root / 'eval_summary.json'
        _safe_copy(eval_summary, export_eval_summary)

export_checkpoints_dir = export_root / 'checkpoints'
if run_dir is not None:
    src_ckpt_dir = run_dir / 'checkpoints'
    if src_ckpt_dir.exists() and src_ckpt_dir.is_dir():
        if export_checkpoints_dir.exists():
            shutil.rmtree(export_checkpoints_dir)
        shutil.copytree(src_ckpt_dir, export_checkpoints_dir)

manifest = {
    'created_at_unix': int(time.time()),
    'run_prefix': rc.run_prefix,
    'runtime_config': export_runtime_cfg.as_posix(),
    'checkpoint_json': export_checkpoint_json.as_posix(),
    'eval_summary': export_eval_summary.as_posix() if export_eval_summary is not None else '',
    'run_dir': run_dir.as_posix() if run_dir is not None else '',
}
(export_root / 'export_manifest.json').write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

print('export_root:', export_root.resolve())
print('export_runtime_cfg:', export_runtime_cfg)
print('export_checkpoint_json:', export_checkpoint_json)
print('export_eval_summary:', export_eval_summary)
print('export_checkpoints_dir:', export_checkpoints_dir if export_checkpoints_dir.exists() else None)